In [6]:
import os
from openai import OpenAI
from mem0 import MemoryClient  
from dotenv import load_dotenv
import hashlib
import os
import tempfile
from pathlib import Path
from datetime import datetime

import streamlit as st
from langchain_core.messages import HumanMessage
from src.agentic_workflow.agent import graph
load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
MEM0_API_KEY = os.environ["MEM0_API_KEY"]

llm_client = OpenAI(api_key=OPENAI_API_KEY)
mem_client = MemoryClient(api_key=MEM0_API_KEY)

c:\dev\accountant_agent_v3\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\dev\accountant_agent_v3\env\Lib\site-packages\typer\__init__.py:24: DeprecationWarning: 'click.utils.get_binary_stream' is deprecated and will be removed in Click 9.0.
  from click.utils import get_binary_stream as get_binary_stream
c:\dev\accountant_agent_v3\env\Lib\site-packages\typer\__init__.py:25: DeprecationWarning: 'click.utils.get_text_stream' is deprecated and will be removed in Click 9.0.
  from click.utils import get_text_stream as get_text_stream
2026-09-27 11:51:19,144 - INFO - embedding get initialized



🚀 RAG Pipeline initialized
📂 Data: ./data/PLAN_COMPTABLE
💾 Persist: artifacts/vectorestore/db_plan_comptable
🔍 Vectorstore exists: True



2026-09-27 11:51:19,987 - INFO - initialiased embeding model OpenAIEmbeddings with model name : text-embedding-3-large
2026-09-27 11:51:19,987 - INFO - ✅ VectorStore initialized with persist_dir: artifacts/vectorestore/db_plan_comptable
2026-09-27 11:51:20,153 - INFO - ✅ Loaded Chroma from artifacts/vectorestore/db_plan_comptable
2026-09-27 11:51:20,157 - INFO - ✅ Loaded BM25 index from artifacts\vectorestore\db_plan_comptable\bm25_index.pkl
2026-09-27 11:51:20,157 - INFO - ✅ Hybrid retriever restored
2026-09-27 11:51:20,177 - INFO - embedding get initialized
2026-09-27 11:51:20,196 - INFO - initialiased embeding model OpenAIEmbeddings with model name : text-embedding-3-large


🔄 Loading existing vectorstore...
✅ Loaded existing vectorstore from artifacts/vectorestore/db_plan_comptable

🚀 RAG Pipeline initialized
📂 Data: ./data/LOIS_DE_FINANCE
💾 Persist: artifacts/vectorestore/LOIS_DE_FINANCE
🔍 Vectorstore exists: True



2026-09-27 11:51:20,196 - INFO - ✅ VectorStore initialized with persist_dir: artifacts/vectorestore/LOIS_DE_FINANCE
2026-09-27 11:51:20,217 - INFO - ✅ Loaded Chroma from artifacts/vectorestore/LOIS_DE_FINANCE
2026-09-27 11:51:20,242 - INFO - ✅ Loaded BM25 index from artifacts\vectorestore\LOIS_DE_FINANCE\bm25_index.pkl
2026-09-27 11:51:20,245 - INFO - ✅ Hybrid retriever restored


🔄 Loading existing vectorstore...
✅ Loaded existing vectorstore from artifacts/vectorestore/LOIS_DE_FINANCE


2026-09-27 11:51:20,406 - INFO - embedding get initialized
2026-09-27 11:51:20,427 - INFO - initialiased embeding model OpenAIEmbeddings with model name : text-embedding-3-large
2026-09-27 11:51:20,428 - INFO - ✅ VectorStore initialized with persist_dir: artifacts/vectorestore/db_CGNC
2026-09-27 11:51:20,447 - INFO - ✅ Loaded Chroma from artifacts/vectorestore/db_CGNC
2026-09-27 11:51:20,460 - INFO - ✅ Loaded BM25 index from artifacts\vectorestore\db_CGNC\bm25_index.pkl
2026-09-27 11:51:20,462 - INFO - ✅ Hybrid retriever restored
2026-09-27 11:51:20,464 - INFO - embedding get initialized
2026-09-27 11:51:20,477 - INFO - initialiased embeding model OpenAIEmbeddings with model name : text-embedding-3-large
2026-09-27 11:51:20,477 - INFO - ✅ VectorStore initialized with persist_dir: artifacts/vectorestore/db_CGI
2026-09-27 11:51:20,504 - INFO - ✅ Loaded Chroma from artifacts/vectorestore/db_CGI
2026-09-27 11:51:20,546 - INFO - ✅ Loaded BM25 index from artifacts\vectorestore\db_CGI\bm25_in


🚀 RAG Pipeline initialized
📂 Data: ./data/CGNC
💾 Persist: artifacts/vectorestore/db_CGNC
🔍 Vectorstore exists: True

🔄 Loading existing vectorstore...
✅ Loaded existing vectorstore from artifacts/vectorestore/db_CGNC

🚀 RAG Pipeline initialized
📂 Data: ./data/CGI
💾 Persist: artifacts/vectorestore/db_CGI
🔍 Vectorstore exists: True

🔄 Loading existing vectorstore...
✅ Loaded existing vectorstore from artifacts/vectorestore/db_CGI


2026-09-27 11:51:21,757 - INFO - HTTP Request: GET https://api.mem0.ai/v1/ping/ "HTTP/1.1 200 OK"


In [32]:
AGENT_ID = "support-agent-v1"

def retrieve_memories(user_id: str, query: str, limit: int = 5):
    """Fetch relevant long-term memories from Mem0."""
    results = mem_client.search(
        user_id=user_id,
        agent_id=AGENT_ID,
        query=query,
        limit=limit,
    )
    return results  # list of dicts with 'text', 'score', 'metadata', etc.

In [33]:
from src.agentic_workflow.agent import graph

config = {
        "configurable": {"thread_id": "default_thread"}
    }

In [7]:
class memory:
    def __init__(self,user_prompt:str,user_id:str,agent_id:str,limit:int=5):

        self.user_prompt = user_prompt
        self.user_id = user_id
        self.limit = limit
        self.agent_id = agent_id

    def retrive(self):
        results = mem_client.search(
                user_id=self.user_id,
                agent_id=self.agent_id,
                query=self.user_prompt,
                limit=self.limit,
            )
        return results
    def store_memory(self):
        """Store a new memory in Mem0."""
        mem_client.add(
            user_id=self.user_id,
            agent_id=self.agent_id,
            text=self.user_prompt,
        )
    def build_prompt(self,memories: list[dict]) -> str:
        memory_block = ""
        if memories:
            context = "\n".join(f"- {m['text']}" for m in memories)
            memory_block = f"Known user context:\n{context}\n\n"

        system = (
            "You are a helpful support agent. Use the known user context when relevant, "
            "and avoid asking for information already present there."
        )
        return f"{system}\n\n{memory_block}User: {self.user_prompt}\nAssistant:"

    def semantic_memory_for_facts(self) -> str | None:
        """Heuristic: store preferences or explicit facts."""
        triggers = ["I prefer", "I usually", "My name is", "I am working on", "I live in"]
        lower_msg = self.user_prompt.lower()
        if any(t.lower() in lower_msg for t in triggers):
            return self.user_prompt
        # optionally parse structured memory using another LLM call
        return None
    

In [35]:
awareness=memory(user_id="ilias", agent_id="agent1")

In [ ]:
prompt="how are you?"
result = graph.invoke({"messages": [HumanMessage(content=prompt)]},config=config)
print(result["messages"][-1].content)

2026-09-27 10:55:45,196 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


hi


2026-09-27 10:56:17,317 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


I'm here and ready to assist you! If you have any questions or need help, feel free to ask. How can I help you today?


In [ ]:
from src.agentic_workflow import agent


def call_llm(prompt: str) -> str:
        result = agent.invoke({"messages": [HumanMessage(content=prompt)]},config=config)
    return result

In [22]:
def store_memory(user_id: str, text: str):
    """Store a new memory in Mem0."""
    mem_client.add(
        user_id=user_id,
        agent_id=AGENT_ID,
        text=text,
    )

In [ ]:
def build_prompt(user_message: str, memories: list[dict]) -> str:
    memory_block = ""
    if memories:
        context = "\n".join(f"- {m['text']}" for m in memories)
        memory_block = f"Known user context:\n{context}\n\n"

    system = (
        "You are a helpful support agent. Use the known user context when relevant, "
        "and avoid asking for information already present there."
    )
    return f"{system}\n\n{memory_block}User: {user_message}\nAssistant:"

In [ ]:
def call_llm(prompt: str) -> str:
    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

In [ ]:
def semantic_memory_for_facts(user_message: str, assistant_reply: str) -> str | None:
    """Heuristic: store preferences or explicit facts."""
    triggers = ["I prefer", "I usually", "My name is", "I am working on", "I live in"]
    lower_msg = user_message.lower()
    if any(t.lower() in lower_msg for t in triggers):
        return user_message
    # optionally parse structured memory using another LLM call
    return None

In [ ]:
def chat_with_memory(user_id: str):
    print("Start chatting. Type 'exit' to quit.")
    while True:
        user_message = input("You: ")
        if user_message.lower() in {"exit", "quit"}:
            break

        # 1) Retrieve relevant memories
        memories = retrieve_memories(user_id=user_id, query=user_message)

        # 2) Build prompt with memory
        prompt = build_prompt(user_message, memories)

        # 3) Call LLM
        assistant_reply = call_llm(prompt)
        print(f"Agent: {assistant_reply}\n")

        # 4) Store new memory when appropriate
        candidate = semantic_memory_for_facts(user_message, assistant_reply)
        if candidate:
            store_memory(
                user_id=user_id,
                text=candidate,
            )

In [8]:
class memory:
    def __init__(self,user_prompt:str,user_id:str,agent_id:str,limit:int=5):
        self.user_prompt = user_prompt
        self.user_id = user_id
        self.limit = limit
        self.agent_id = agent_id

    def retrive(self):
        results = mem_client.search(
                filters={'user_id': self.user_id, 'agent_id': self.agent_id},
                query=self.user_prompt,
                limit=self.limit,
            )
        return results
    def store_memory(self,memory_data: dict[str, str]):
        """Store a new memory in Mem0."""
        mem_client.add(
            user_id=memory_data["user_id"],
        agent_id=memory_data["agent_id"],
            messages=[
            {
                "role": "user",
                "content": memory_data["text"],
            },
            {
                "role": "assistant",
                "content": memory_data["llm_output"],
            },
        ],
        )
    def build_prompt(self,memories: list[dict]) -> str:
        memory_block = ""
        if memories:
            context = "\n".join(f"- {m}" for m in memories)
            memory_block = f"Known user context:\n{context}\n\n"

        system = (
            "You are a helpful support agent. Use the known user context when relevant, "
            "and avoid asking for information already present there."
        )
        return f"{system}\n\n{memory_block}User: {self.user_prompt}\nAssistant:"

    def semantic_memory_for_facts(self,llm_output: str) -> dict [str, str] | None:
        """Heuristic: store preferences or explicit facts."""
        triggers = ["I prefer", "I usually", "My name is", "I am working on", "I live in"]
        lower_msg = self.user_prompt.lower()
        if any(t.lower() in lower_msg for t in triggers):
            return {"user_id": self.user_id, "agent_id": self.agent_id, "text": self.user_prompt,"llm_output": llm_output,"timestamp": datetime.now().isoformat()}
        # optionally parse structured memory using another LLM call
        return None
    

In [12]:
config = {
        "configurable": {"thread_id": "default_thread"}
    }

In [16]:
prompt="my name is ilias and I prefer to work on accounting tasks."

In [17]:
m=memory(user_prompt=prompt, user_id="ilias", agent_id="agent1")

In [19]:
long_tm=m.retrive()
build_prompt=m.build_prompt(long_tm)
agent_response = graph.invoke({"messages": [HumanMessage(content=build_prompt)]},config=config)
x=m.semantic_memory_for_facts(agent_response["messages"][-1].content)
print(agent_response["messages"][-1].content)
if x:
    m.store_memory(x)

2026-09-27 11:53:18,657 - INFO - HTTP Request: POST https://api.mem0.ai/v3/memories/search/ "HTTP/1.1 200 OK"
2026-09-27 11:53:19,767 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Thank you for sharing, Ilias! How can I assist you with your accounting tasks today? If you have any specific questions or need help with something, feel free to let me know!


2026-09-27 11:53:20,222 - INFO - HTTP Request: POST https://api.mem0.ai/v3/memories/add/ "HTTP/1.1 200 OK"


## memory fix problemes

In [ ]:
import os
from mem0 import MemoryClient  
from dotenv import load_dotenv

from datetime import datetime


load_dotenv()

MEM0_API_KEY = os.environ["MEM0_API_KEY"]

mem_client = MemoryClient(api_key=MEM0_API_KEY)


class Memory:
    def __init__(self,user_prompt:str,user_id:str,agent_id:str,limit:int=5):
        self.user_prompt = user_prompt
        self.user_id = user_id
        self.limit = limit
        self.agent_id = agent_id

    def retrive(self):
        results = mem_client.search(
                filters={'user_id': self.user_id, 'agent_id': self.agent_id},
                query=self.user_prompt,
                limit=self.limit,
            )
        return results
    def store_memory(self,memory_data: dict[str, str]):
        """Store a new memory in Mem0."""
        mem_client.add(
            user_id=memory_data["user_id"],
        agent_id=memory_data["agent_id"],
            messages=[
            {
                "role": "user",
                "content": memory_data["text"],
            },
            {
                "role": "assistant",
                "content": memory_data["llm_output"],
            },
        ],
        )
    def _build_prompt(self,memories: list[dict]) -> str:
        """unused function, only keeped as reference from documentation"""
        memory_block = ""
        if memories:
            context = "\n".join(f"- {m}" for m in memories)
            memory_block = f"Known user context:\n{context}\n\n"

        system = (
            "You are a helpful support agent. Use the known user context when relevant, "
            "and avoid asking for information already present there."
        )
        return f"{system}\n\n{memory_block}User: {self.user_prompt}\nAssistant:"

    def semantic_memory_for_facts(self,llm_output: str) -> dict [str, str] | None:
        return {"user_id": self.user_id, "agent_id": self.agent_id, "text": self.user_prompt,"llm_output": llm_output,"timestamp": datetime.now().isoformat()}
    

2026-09-27 13:14:03,482 - INFO - HTTP Request: GET https://api.mem0.ai/v1/ping/ "HTTP/1.1 200 OK"


In [21]:
user_text="hi"

In [23]:
memory=Memory(user_prompt=user_text, user_id="ilias", agent_id="agent1")
long_tm=memory.retrive()
build_prompt=memory.build_prompt(long_tm)
agent_response = graph.invoke({"messages": [HumanMessage(content=build_prompt)]},config=config)
x=memory.semantic_memory_for_facts(agent_response["messages"][-1].content)
if x:
    memory.store_memory(x)
agent_response["messages"][-1].content

2026-09-27 13:14:49,029 - INFO - HTTP Request: POST https://api.mem0.ai/v3/memories/search/ "HTTP/1.1 200 OK"
2026-09-27 13:14:51,478 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-27 13:14:51,985 - INFO - HTTP Request: POST https://api.mem0.ai/v3/memories/add/ "HTTP/1.1 200 OK"


'Hello, Ilias! How can I assist you with your accounting tasks today?'